# 60 - Benchmark: RAF-DB Dataset (Self Train-Test)

**Dataset:** RAF-DB — 14,449 gambar basic-emotion (11,565 train / 2,884 test) dari Kaggle shuvoalok.
**Evaluasi:** Official train/test split (dari paper original).
**Model:** Semua model + Late Fusion (CNN, FCNN, Intermediate, CNN_TL, Intermediate_TL, Late_Fusion).
**Skenario:** B1 (Baseline) — tidak pakai class weights / augmentasi agar konsisten dengan benchmark standar.
**Metrik:** Macro F1, Micro F1 (=accuracy), Weighted F1 (semua di-print).

In [ ]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
)
from training.utils import train_model, full_evaluation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
OUTPUT_DIR = PROJECT_ROOT / 'models' / 'benchmark' / 'rafdb'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = [
    ('CNN', EmotionCNN, 'cnn', 0.0001),
    ('FCNN', EmotionFCNN, 'fcnn', 0.0001),
    ('Intermediate', IntermediateFusion, 'fusion', 0.0001),
    ('CNN_TL', EmotionCNNTransfer, 'cnn', 0.00005),
    ('Intermediate_TL', IntermediateFusionTransfer, 'fusion', 0.00005),
]

print('Setup complete.')

In [ ]:
# ── Helper Functions ──

def make_loader(images, landmarks, labels, model_type, batch_size=32, shuffle=True):
    img_t = torch.from_numpy(images).permute(0, 3, 1, 2)
    lm_t = torch.from_numpy(landmarks)
    y_t = torch.from_numpy(labels).long()
    if model_type == 'cnn': ds = TensorDataset(img_t, y_t)
    elif model_type == 'fcnn': ds = TensorDataset(lm_t, y_t)
    else: ds = TensorDataset(img_t, lm_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


def metrics_triple(y_true, y_pred):
    return {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
    }


def train_and_eval(ModelClass, model_type, lr,
                   train_img, train_lm, train_y,
                   val_img, val_lm, val_y,
                   test_img, test_lm, test_y,
                   emotions, save_dir):
    tr_loader = make_loader(train_img, train_lm, train_y, model_type, BATCH_SIZE)
    val_loader = make_loader(val_img, val_lm, val_y, model_type, BATCH_SIZE, False)
    test_loader = make_loader(test_img, test_lm, test_y, model_type, BATCH_SIZE, False)

    model = ModelClass(num_classes=len(emotions)).to(device)
    save_path = str(save_dir / 'model.pth')
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8, min_lr=1e-7)

    train_model(model, tr_loader, val_loader, criterion, optimizer, scheduler,
                device, model_type, EPOCHS, PATIENCE, save_path)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    r = full_evaluation(model, test_loader, criterion, device, model_type, emotions)
    return {
        'accuracy': float(r['test_accuracy']),
        'macro_f1': float(r['test_macro_f1']),
        'micro_f1': float(r['test_micro_f1']),
        'weighted_f1': float(r['test_weighted_f1']),
    }


def late_fusion_eval(train_img, train_lm, train_y,
                     val_img, val_lm, val_y,
                     test_img, test_lm, test_y,
                     num_classes, save_dir):
    cnn = EmotionCNN(num_classes=num_classes).to(device)
    tr_cnn = make_loader(train_img, train_lm, train_y, 'cnn', BATCH_SIZE)
    val_cnn = make_loader(val_img, val_lm, val_y, 'cnn', BATCH_SIZE, False)
    opt = torch.optim.Adam(cnn.parameters(), lr=0.0001)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=8, min_lr=1e-7)
    train_model(cnn, tr_cnn, val_cnn, nn.CrossEntropyLoss(), opt, sch,
                device, 'cnn', EPOCHS, PATIENCE, str(save_dir / 'cnn.pth'))

    fcnn = EmotionFCNN(num_classes=num_classes).to(device)
    tr_fcnn = make_loader(train_img, train_lm, train_y, 'fcnn', BATCH_SIZE)
    val_fcnn = make_loader(val_img, val_lm, val_y, 'fcnn', BATCH_SIZE, False)
    opt2 = torch.optim.Adam(fcnn.parameters(), lr=0.0001)
    sch2 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='max', factor=0.5, patience=8, min_lr=1e-7)
    train_model(fcnn, tr_fcnn, val_fcnn, nn.CrossEntropyLoss(), opt2, sch2,
                device, 'fcnn', EPOCHS, PATIENCE, str(save_dir / 'fcnn.pth'))

    cnn.load_state_dict(torch.load(save_dir / 'cnn.pth', map_location=device, weights_only=True))
    fcnn.load_state_dict(torch.load(save_dir / 'fcnn.pth', map_location=device, weights_only=True))
    cnn.eval(); fcnn.eval()

    # Get val probs for weight tuning
    val_img_t = torch.from_numpy(val_img).permute(0, 3, 1, 2).to(device)
    val_lm_t = torch.from_numpy(val_lm).to(device)
    with torch.no_grad():
        vc = torch.softmax(cnn(val_img_t), dim=1).cpu().numpy()
        vf = torch.softmax(fcnn(val_lm_t), dim=1).cpu().numpy()

    best_wf1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        preds = (w * vc + (1-w) * vf).argmax(axis=1)
        f = f1_score(val_y, preds, average='macro', zero_division=0)
        if f > best_wf1: best_wf1 = f; best_w = w
    print(f'    Best late-fusion weight (by val macro F1): cnn={best_w:.2f}')

    test_img_t = torch.from_numpy(test_img).permute(0, 3, 1, 2).to(device)
    test_lm_t = torch.from_numpy(test_lm).to(device)
    with torch.no_grad():
        cnn_probs = torch.softmax(cnn(test_img_t), dim=1).cpu().numpy()
        fcnn_probs = torch.softmax(fcnn(test_lm_t), dim=1).cpu().numpy()
    preds = (best_w * cnn_probs + (1-best_w) * fcnn_probs).argmax(axis=1)
    m = metrics_triple(test_y, preds)
    m['best_cnn_weight'] = float(best_w)
    return m


def run_benchmark(num_classes, emotions, data_dir):
    print(f"\n{'='*70}")
    print(f'  RAF-DB {num_classes}-class (Official train/test split, B1)')
    print(f"{'='*70}")

    X_tr = np.load(data_dir / 'X_train_images.npy')
    L_tr = np.load(data_dir / 'X_train_landmarks.npy')
    y_tr = np.load(data_dir / 'y_train.npy')
    X_te = np.load(data_dir / 'X_test_images.npy')
    L_te = np.load(data_dir / 'X_test_landmarks.npy')
    y_te = np.load(data_dir / 'y_test.npy')

    # 10% of train -> val (stratified)
    from sklearn.model_selection import train_test_split
    idx_tr, idx_va = train_test_split(np.arange(len(y_tr)), test_size=0.1, stratify=y_tr, random_state=42)
    print(f'  Train: {len(idx_tr)}  Val: {len(idx_va)}  Test: {len(y_te)}')

    all_results = {}
    models_to_run = MODELS + [('Late_Fusion', None, 'late', 0.0001)]

    for model_name, ModelClass, model_type, lr in models_to_run:
        key = f'{model_name}_B1'
        print(f'\n  >> {key} ...')
        save_dir = OUTPUT_DIR / f'{num_classes}c' / key
        os.makedirs(save_dir, exist_ok=True)

        tr_img, tr_lm, tr_y = X_tr[idx_tr], L_tr[idx_tr], y_tr[idx_tr]
        v_img, v_lm, v_y = X_tr[idx_va], L_tr[idx_va], y_tr[idx_va]

        if model_type == 'late':
            r = late_fusion_eval(tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                                  X_te, L_te, y_te, num_classes, save_dir)
        else:
            r = train_and_eval(ModelClass, model_type, lr,
                                tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                                X_te, L_te, y_te, emotions, save_dir)
        all_results[key] = r
        print(f"    Macro={r['macro_f1']:.4f}  Micro={r['micro_f1']:.4f}  Weighted={r['weighted_f1']:.4f}")

    save_path = OUTPUT_DIR / f'rafdb_{num_classes}c_results.json'
    with open(save_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'\n  Saved: {save_path}')
    return all_results

print('Helper functions ready.')

## Run RAF-DB Benchmark

In [ ]:
BENCHMARK_DIR = PROJECT_ROOT / 'data' / 'benchmark'
EMOTIONS_7 = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgusted', 'surprised']
EMOTIONS_4 = ['neutral', 'happy', 'sad', 'negative']

res_7c = run_benchmark(7, EMOTIONS_7, BENCHMARK_DIR / 'rafdb_7class')
res_4c = run_benchmark(4, EMOTIONS_4, BENCHMARK_DIR / 'rafdb_4class')

## Ringkasan RAF-DB (Semua Metrik)

In [ ]:
for nc_label, res in [('7-class', res_7c), ('4-class', res_4c)]:
    print(f"\n{'='*78}")
    print(f'  RAF-DB {nc_label} - sorted by Macro F1')
    print(f"{'='*78}")
    print(f"  {'Model':<22} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Accuracy':>10}")
    print(f"  {'-'*70}")
    for key in sorted(res.keys(), key=lambda k: -res[k]['macro_f1']):
        r = res[key]
        print(f"  {key:<22} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} {r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f}")